# Clase 07 - Calidad del dato y Data Profiling

Esta notebook acompaña la documentación de la Clase 07. Vamos a diagnosticar un conjunto ficticio de ventas antes de transformarlo: completitud, exactitud, consistencia, unicidad, vigencia e integridad; profiling; KPIs; valores atípicos con IQR; y un contrato de calidad.

**Objetivo:** que cada operación se vea y se ejecute aquí mismo. Las funciones que usamos se definen en las celdas de esta notebook; no dependemos de `gobierno_demo.py`.

**Entorno:** Python local y pandas. Great Expectations se incluye como demostración opcional porque no forma parte de las dependencias base de la Unidad III. Si no está instalado, ejecutaremos el mismo contrato con pandas para que toda la clase pueda avanzar sin una instalación adicional.

Los datos son ficticios. Ejecuta las celdas de arriba hacia abajo.

## 1. Las seis dimensiones de calidad

| Dimensión | Pregunta | Ejemplo de la documentación |
|---|---|---|
| **Completitud** | ¿Están presentes los datos necesarios? | Un email nulo cuando es obligatorio para una campaña. |
| **Exactitud** | ¿El dato representa la realidad? | Un monto que se contrasta con una fuente verificada. |
| **Consistencia** | ¿Se respetan reglas entre valores y sistemas? | `ARS`, `ars` y `PESOS` para expresar una moneda. |
| **Unicidad** | ¿Cada entidad aparece una sola vez? | Un `id_venta` repetido que duplica una transacción. |
| **Vigencia** | ¿El dato está actualizado para su uso? | Un domicilio o precio que quedó desactualizado. |
| **Integridad** | ¿Las relaciones y dominios son válidos? | Un `id_cliente` debe existir en el maestro de clientes. |

Las dimensiones describen preguntas diferentes, pero un registro puede incumplir varias. Un valor presente no necesariamente es exacto; para evaluar exactitud hace falta compararlo con una fuente de verdad. En este ejemplo, algunas reglas de negocio son didácticas y no sustituyen la validación con información real.

## 2. Preparar los datos de práctica

El conjunto conserva los problemas intencionales de la Clase 07: un ID repetido y uno nulo, una fecha imposible, monedas con formatos distintos, montos negativos o extremos y emails nulos o mal formados. Guardamos `df_crudo` y trabajamos con `df` para no perder la entrada original.

In [1]:
import pandas as pd  # Importa pandas para trabajar con tablas.
import numpy as np  # Importa NumPy para operaciones numéricas.
import re  # Importa expresiones regulares para revisar emails.

data = {  # Guarda en un diccionario las columnas del ejemplo.
    "id_venta": [1, 2, 2, 3, None, 5, 6, 7, 8, 9],  # Incluye un ID repetido y uno faltante.
    "fecha_venta": ["2025-01-05", "2025-01-06", "2025-01-06", "2025-99-01", "2025-01-08", "2025-01-09", "2025-01-10", "2025-01-11", "2025-01-12", "2025-01-13"],  # Incluye una fecha imposible.
    "id_cliente": [101, 102, 102, 103, 104, 105, 9999, 107, 108, 109],  # El cliente 9999 no tiene referencia aquí.
    "monto": [1500.0, 2300.0, 2300.0, 850.0, 1200.0, -50.0, 950.0, 0.0, 1800.0, 150000.0],  # Incluye montos no positivos y un valor extremo.
    "moneda": ["ARS", "ars", "ars", "ARS", "PESOS", "USD", "ARS", "EUR", "ARS", "ARS"],  # Incluye diferencias de formato y un dominio inválido.
    "email_cliente": ["a@b.com", "juan@@err.com", "juan@@err.com", None, "ana@test.com", None, "luis@ok.com", "carlos@ok.com", None, "pedro@ok.com"],  # Incluye nulos y formatos inválidos.
}  # Termina la definición de los datos del ejemplo.
df_crudo = pd.DataFrame(data)  # Convierte el diccionario en una tabla.
df = df_crudo.copy()  # Conserva el origen y crea una tabla de trabajo.
df["fecha_venta"] = pd.to_datetime(df["fecha_venta"], errors="coerce")  # Convierte fechas inválidas en NaT para poder detectarlas.
display(df)  # Muestra las diez filas luego de interpretar las fechas.
print("Filas:", len(df), "| Columnas:", len(df.columns))  # Resume el tamaño del conjunto.

,id_venta,fecha_venta,id_cliente,monto,moneda,email_cliente
0,1.0,2025-01-05,101,1500.0,ARS,a@b.com
1,2.0,2025-01-06,102,2300.0,ars,juan@@err.com
2,2.0,2025-01-06,102,2300.0,ars,juan@@err.com
3,3.0,NaT,103,850.0,ARS,NaN
4,NaN,2025-01-08,104,1200.0,PESOS,ana@test.com
5,5.0,2025-01-09,105,-50.0,USD,NaN
6,6.0,2025-01-10,9999,950.0,ARS,luis@ok.com
7,7.0,2025-01-11,107,0.0,EUR,carlos@ok.com
8,8.0,2025-01-12,108,1800.0,ARS,NaN
9,9.0,2025-01-13,109,150000.0,ARS,pedro@ok.com


Filas: 10 | Columnas: 6


## 3. Data Profiling: diagnosticar antes de transformar

Primero observamos la estructura y los problemas. La completitud de esta tabla considera faltantes tanto los nulos como las fechas imposibles que pandas convirtió en `NaT`. Un valor repetido en una columna no siempre significa que la fila completa esté duplicada.

In [2]:
def generar_reporte_profiling(tabla):  # Define el reporte de diagnóstico de la tabla.
    total_filas = len(tabla)  # Cuenta las filas para calcular porcentajes.
    nulos = tabla.isna().sum()  # Cuenta los valores nulos por columna.
    porcentaje_nulos = (nulos / total_filas * 100).round(1)  # Convierte los conteos a porcentajes.
    reporte = pd.DataFrame({"Nulos": nulos, "% nulos": porcentaje_nulos, "% completo": (100 - porcentaje_nulos).round(1)})  # Resume completitud por columna.
    duplicados = int(tabla.duplicated().sum())  # Cuenta filas idénticas en todas sus columnas.
    print("Filas:", total_filas, "| Columnas:", len(tabla.columns))  # Muestra el tamaño del conjunto.
    print("Filas completamente duplicadas:", duplicados)  # Informa los duplicados exactos.
    display(reporte)  # Muestra la tabla de completitud.
    columnas_numericas = tabla.select_dtypes(include="number")  # Selecciona las columnas numéricas.
    if not columnas_numericas.empty:  # Comprueba que haya columnas numéricas para resumir.
        print("Estadísticas de columnas numéricas:")  # Anuncia el resumen estadístico.
        display(columnas_numericas.describe().T)  # Muestra conteos, medias, mínimos y máximos.
    for columna in tabla.columns:  # Recorre las columnas para localizar las que contienen texto.
        es_texto = pd.api.types.is_string_dtype(tabla[columna]) or pd.api.types.is_object_dtype(tabla[columna])  # Reconoce texto con tipos de pandas distintos.
        if es_texto:  # Omite columnas que no contienen texto.
            print("Valores de", columna)  # Identifica la columna que se está inspeccionando.
            display(tabla[columna].value_counts(dropna=False).rename("frecuencia").to_frame())  # Cuenta categorías e incluye nulos.
    return reporte  # Devuelve la tabla para reutilizarla.

reporte_profiling = generar_reporte_profiling(df)  # Ejecuta el diagnóstico sobre los datos.
print("Fechas no interpretables:", int(df["fecha_venta"].isna().sum()))  # Cuenta las fechas convertidas en NaT.

Filas: 10 | Columnas: 6
Filas completamente duplicadas: 1


,Nulos,% nulos,% completo
id_venta,1,10.0,90.0
fecha_venta,1,10.0,90.0
id_cliente,0,0.0,100.0
monto,0,0.0,100.0
moneda,0,0.0,100.0
email_cliente,3,30.0,70.0


Estadísticas de columnas numéricas:


,count,mean,std,min,25%,50%,75%,max
id_venta,9.0,4.777778,2.905933,1.0,2.00,5.0,7.00,9.0
id_cliente,10.0,1094.000000,3128.899238,101.0,102.25,104.5,107.75,9999.0
monto,10.0,16085.000000,47060.086001,-50.0,875.00,1350.0,2175.00,150000.0


Valores de moneda


,frecuencia
moneda,
ARS,5
ars,2
PESOS,1
USD,1
EUR,1


Valores de email_cliente


,frecuencia
email_cliente,
NaN,3
juan@@err.com,2
a@b.com,1
ana@test.com,1
luis@ok.com,1
carlos@ok.com,1
pedro@ok.com,1


Fechas no interpretables: 1


## 4. Métricas y KPIs de calidad

Calcularemos indicadores sencillos sobre completitud, unicidad, consistencia del dominio de monedas y reglas del monto, además del formato de email. Son métricas de este ejercicio: la exactitud real y la vigencia necesitan referencias y fechas de corte definidas por el negocio.

In [3]:
total = len(df)  # Guarda el total de filas para usarlo como denominador.
print("COMPLETITUD")  # Separa visualmente los indicadores de completitud.
print("id_venta:", round(df["id_venta"].notna().mean() * 100, 1), "%")  # Calcula qué porcentaje tiene ID.
print("monto:", round(df["monto"].notna().mean() * 100, 1), "%")  # Calcula qué porcentaje tiene monto.
print("email_cliente:", round(df["email_cliente"].notna().mean() * 100, 1), "%")  # Calcula qué porcentaje tiene email.
print("fecha_venta:", round(df["fecha_venta"].notna().mean() * 100, 1), "%")  # Cuenta NaT como fecha faltante.
duplicados_id = int(df["id_venta"].duplicated().sum())  # Cuenta IDs repetidos después de su primera aparición.
print("\nUNICIDAD")  # Anuncia el indicador de unicidad.
print("IDs de venta repetidos:", duplicados_id)  # Muestra la cantidad de IDs repetidos.
print("Unicidad de id_venta:", round((total - duplicados_id) / total * 100, 1), "%")  # Calcula unicidad sobre el total de filas.
monedas_validas = {"ARS", "USD", "EUR"}  # Define el dominio permitido por este ejercicio.
monedas_normalizadas = df["moneda"].str.strip().str.upper()  # Quita espacios y convierte monedas a mayúsculas.
monedas_invalidas = int((~monedas_normalizadas.isin(monedas_validas)).sum())  # Cuenta valores fuera del dominio.
print("\nCONSISTENCIA DE MONEDA")  # Anuncia la revisión de consistencia.
print("Monedas fuera del dominio:", monedas_invalidas)  # Muestra las monedas inválidas.
print("Consistencia de moneda:", round((total - monedas_invalidas) / total * 100, 1), "%")  # Calcula filas dentro del dominio.
montos_invalidos = int((df["monto"] <= 0).sum())  # Cuenta montos cero o negativos según la regla de negocio.
print("\nREGLA DE MONTO")  # Anuncia la revisión de montos.
print("Montos menores o iguales a cero:", montos_invalidos)  # Muestra cuántos montos incumplen la regla.
print("Cumplimiento del monto:", round((total - montos_invalidos) / total * 100, 1), "%")  # Calcula el porcentaje que cumple.
patron_email = r"^[a-zA-Z0-9._%+\-]+@[a-zA-Z0-9.\-]+\.[a-zA-Z]{2,}$"  # Define un patrón didáctico para emails.
emails_no_nulos = df["email_cliente"].dropna()  # Excluye nulos para medir solamente el formato.
emails_validos = int(emails_no_nulos.str.match(patron_email).sum())  # Cuenta los emails que cumplen el patrón.
print("\nFORMATO DE EMAIL")  # Anuncia la revisión del formato.
print("Emails no nulos:", len(emails_no_nulos))  # Informa cuántos emails se pudieron revisar.
print("Emails con formato válido:", emails_validos)  # Informa cuántos cumplen el patrón.
print("Formato válido:", round(emails_validos / len(emails_no_nulos) * 100, 1), "%")  # Calcula validez entre emails presentes.
scores = [df["id_venta"].notna().mean(), (total - duplicados_id) / total, df["monto"].notna().mean(), df["email_cliente"].notna().mean(), (total - monedas_invalidas) / total, (total - montos_invalidos) / total]  # Reúne los seis indicadores usados para el score.
score_global = float(np.mean(scores) * 100)  # Promedia los indicadores sin ponderación.
estado_score = "Aceptable" if score_global >= 90 else "Revisar" if score_global >= 75 else "Crítico"  # Clasifica el resultado con umbrales del ejemplo.
print("\nSCORE GLOBAL:", round(score_global, 1), "% -", estado_score)  # Presenta el score didáctico.

COMPLETITUD
id_venta: 90.0 %
monto: 100.0 %
email_cliente: 70.0 %
fecha_venta: 90.0 %

UNICIDAD
IDs de venta repetidos: 1
Unicidad de id_venta: 90.0 %

CONSISTENCIA DE MONEDA
Monedas fuera del dominio: 1
Consistencia de moneda: 90.0 %

REGLA DE MONTO
Montos menores o iguales a cero: 2
Cumplimiento del monto: 80.0 %

FORMATO DE EMAIL
Emails no nulos: 7
Emails con formato válido: 5
Formato válido: 71.4 %

SCORE GLOBAL: 86.7 % - Revisar


## 5. Detectar valores atípicos con IQR

El rango intercuartílico se calcula como `Q3 - Q1`. Marcamos como posibles atípicos los valores menores que `Q1 - 1.5 × IQR` o mayores que `Q3 + 1.5 × IQR`. Es una señal para investigar, no una orden automática para borrar el registro. La regla de negocio de monto positivo se evalúa por separado.

In [4]:
montos = pd.Series([1500.0, 2300.0, 850.0, 1200.0, 950.0, 1800.0, -50.0, 150000.0, 1600.0, 1100.0], name="monto")  # Prepara los montos del ejemplo de IQR.

def detectar_outliers_iqr(serie):  # Define una función para marcar valores atípicos posibles.
    q1 = serie.quantile(0.25)  # Calcula el primer cuartil.
    q3 = serie.quantile(0.75)  # Calcula el tercer cuartil.
    iqr = q3 - q1  # Calcula el rango intercuartílico.
    limite_inferior = q1 - 1.5 * iqr  # Calcula el límite inferior para investigar.
    limite_superior = q3 + 1.5 * iqr  # Calcula el límite superior para investigar.
    es_outlier = (serie < limite_inferior) | (serie > limite_superior)  # Marca valores fuera de los límites.
    resultado = pd.DataFrame({"valor": serie, "es_outlier": es_outlier})  # Organiza valores y marcas en una tabla.
    resultado["motivo"] = np.where(serie < limite_inferior, "Demasiado bajo", np.where(serie > limite_superior, "Demasiado alto", "Normal"))  # Explica la razón de cada marca.
    print("Q1:", round(q1, 1), "| Q3:", round(q3, 1), "| IQR:", round(iqr, 1))  # Muestra los cuartiles y el rango.
    print("Rango de referencia:", round(limite_inferior, 1), "a", round(limite_superior, 1))  # Muestra los límites calculados.
    return resultado  # Devuelve una fila por monto.

resultado_iqr = detectar_outliers_iqr(montos)  # Ejecuta la detección sobre los montos.
display(resultado_iqr)  # Muestra todos los montos y su clasificación.
print("Posibles outliers:", int(resultado_iqr["es_outlier"].sum()), "de", len(resultado_iqr))  # Resume cuántos requieren revisión.

Q1: 987.5 | Q3: 1750.0 | IQR: 762.5
Rango de referencia: -156.2 a 2893.8


,valor,es_outlier,motivo
0,1500.0,False,Normal
1,2300.0,False,Normal
2,850.0,False,Normal
3,1200.0,False,Normal
4,950.0,False,Normal
5,1800.0,False,Normal
6,-50.0,False,Normal
7,150000.0,True,Demasiado alto
8,1600.0,False,Normal
9,1100.0,False,Normal


Posibles outliers: 1 de 10


## 6. Contrato de calidad con expectativas

Un contrato convierte reglas acordadas en verificaciones repetibles. Primero lo ejecutamos con pandas, disponible en el entorno del curso. Después intentamos ejecutar las mismas expectativas con Great Expectations si está instalado y expone la API de la documentación. La versión opcional no bloquea el resto de la notebook.

In [5]:
datos_contrato = {  # Define un conjunto pequeño para probar las expectativas.
    "id_venta": [1, 2, 2, 3, None, 5],  # Incluye un ID repetido y otro faltante.
    "precio_unitario": [1500.0, 2300.0, 2300.0, -50.0, 850.0, 75000.0],  # Incluye valores fuera del rango esperado.
    "moneda": ["ARS", "ars", "ARS", "ARS", "USD", "BTC"],  # Incluye un formato no normalizado y un dominio inválido.
    "email": ["a@b.com", "juan@@err", "juan@@err", None, "ok@test.com", "otro@ok.com"],  # Incluye nulos y emails con formato incorrecto.
    "cantidad": [1, 2, 2, 3, 0, 1],  # Incluye una cantidad no permitida.
}  # Termina la definición del conjunto de contrato.
df_contrato = pd.DataFrame(datos_contrato)  # Convierte el diccionario de ejemplo en tabla.

def evaluar_contrato(tabla):  # Define las verificaciones del contrato con pandas.
    emails_presentes = tabla["email"].dropna()  # Excluye nulos para medir el formato de emails existentes.
    emails_validos = emails_presentes.str.match(patron_email)  # Comprueba cada email con el patrón definido.
    resultados = {  # Guarda el resultado booleano de cada expectativa.
        "id_venta sin nulos": bool(tabla["id_venta"].notna().all()),  # Exige un identificador en cada fila.
        "precio_unitario sin nulos": bool(tabla["precio_unitario"].notna().all()),  # Exige un precio en cada fila.
        "id_venta único": bool(tabla["id_venta"].is_unique),  # Comprueba que no haya IDs repetidos.
        "precio entre 0.01 y 50000": bool(tabla["precio_unitario"].between(0.01, 50000.0).all()),  # Comprueba el rango de precios.
        "cantidad entre 1 y 9999": bool(tabla["cantidad"].between(1, 9999).all()),  # Comprueba el rango de cantidades.
        "moneda dentro del dominio": bool(tabla["moneda"].isin(["ARS", "USD", "EUR"]).all()),  # Comprueba los códigos de moneda permitidos.
        "formato de email >= 90%": bool(emails_validos.mean() >= 0.9) if len(emails_presentes) else False,  # Exige al menos 90% de los emails presentes válidos.
    }  # Termina la lista de expectativas.
    return pd.DataFrame([{"expectativa": nombre, "aprobada": aprobada} for nombre, aprobada in resultados.items()])  # Devuelve una fila por expectativa.

resultado_contrato = evaluar_contrato(df_contrato)  # Ejecuta todas las reglas definidas en la función.
display(df_contrato)  # Muestra los datos que serán validados.
display(resultado_contrato)  # Muestra qué expectativas se cumplen y cuáles fallan.
print("Expectativas aprobadas:", int(resultado_contrato["aprobada"].sum()), "de", len(resultado_contrato))  # Resume el resultado del contrato.

,id_venta,precio_unitario,moneda,email,cantidad
0,1.0,1500.0,ARS,a@b.com,1
1,2.0,2300.0,ars,juan@@err,2
2,2.0,2300.0,ARS,juan@@err,2
3,3.0,-50.0,ARS,NaN,3
4,NaN,850.0,USD,ok@test.com,0
5,5.0,75000.0,BTC,otro@ok.com,1


,expectativa,aprobada
0,id_venta sin nulos,False
1,precio_unitario sin nulos,True
2,id_venta único,False
3,precio entre 0.01 y 50000,False
4,cantidad entre 1 y 9999,False
5,moneda dentro del dominio,False
6,formato de email >= 90%,False


Expectativas aprobadas: 1 de 7


In [7]:
try:  # Intenta cargar la biblioteca opcional usada en la documentación.
    import great_expectations as ge  # Importa Great Expectations si está instalado.
except ModuleNotFoundError as error:  # Distingue el paquete ausente de una dependencia rota.
    if error.name != "great_expectations":  # Detecta errores de importación dentro del paquete o de sus dependencias.
        raise  # Expone el error real en lugar de ocultarlo como una instalación faltante.
    ge = None  # Marca Great Expectations como no disponible.
if ge is None:  # Comprueba si falta la dependencia opcional.
    print("Great Expectations no está instalado; el contrato con pandas ya se ejecutó arriba.")  # Informa que las reglas locales sí fueron probadas.
elif not hasattr(ge, "from_pandas"):  # Comprueba que la versión instalada tenga la API de la documentación.
    print("La versión instalada usa otra API; consulta la documentación de esa versión.")  # Evita una llamada incompatible.
else:  # Ejecuta Great Expectations cuando la API esperada está disponible.
    validador = ge.from_pandas(df_contrato)  # Crea el validador a partir de la tabla.
    validador.expect_column_values_to_not_be_null("id_venta")  # Exige IDs presentes.
    validador.expect_column_values_to_not_be_null("precio_unitario")  # Exige precios presentes.
    validador.expect_column_values_to_be_unique("id_venta")  # Exige IDs únicos.
    validador.expect_column_values_to_be_between("precio_unitario", min_value=0.01, max_value=50000.0)  # Exige precios dentro del rango.
    validador.expect_column_values_to_be_between("cantidad", min_value=1, max_value=9999)  # Exige cantidades dentro del rango.
    validador.expect_column_values_to_be_in_set("moneda", value_set=["ARS", "USD", "EUR"])  # Exige monedas del dominio permitido.
    validador.expect_column_values_to_match_regex("email", regex=patron_email, mostly=0.9)  # Exige formato válido para al menos 90% de los emails.
    resultado_ge = validador.validate()  # Ejecuta las expectativas sobre los datos.
    estadisticas_ge = resultado_ge["statistics"]  # Recupera el resumen de resultados.
    print("Expectativas aprobadas:", estadisticas_ge["successful_expectations"])  # Muestra las expectativas aprobadas.
    print("Expectativas fallidas:", estadisticas_ge["unsuccessful_expectations"])  # Muestra las expectativas que requieren atención.
    print("Porcentaje aprobado:", round(estadisticas_ge["success_percent"], 1), "%")  # Muestra el porcentaje de expectativas aprobadas.

Expectativas aprobadas: 1
Expectativas fallidas: 6
Porcentaje aprobado: 14.3 %


## Cierre: interpretar antes de corregir

Un resultado fallido señala qué regla investigar; no autoriza a inventar o borrar valores. Los montos negativos, monedas desconocidas y registros repetidos necesitan una decisión del negocio y evidencia. Los outliers también pueden ser operaciones legítimas. La exactitud exige una fuente de verdad y la vigencia una fecha de corte.

**Para conversar:** ¿qué regla cambiaría si el email fuera obligatorio para marketing? ¿Quién debería aprobar ese cambio? En la notebook de contrato trabajaremos finalidad, responsables y versionado.

**Resultado esperado:** 10 ventas; 1 ID repetido; 1 fecha no interpretable; 1 moneda fuera del dominio; 2 montos no positivos. El IQR marca el monto extremo de 150000 como posible atípico; la clasificación depende de los cuartiles calculados.

La práctica sigue la Clase 07 y no certifica calidad ni cumplimiento normativo. Las definiciones de umbral y uso deben aprobarse en el gobierno del dato.